# Lab | Music recommendations

- First re-run everything in this notebook to ensure you're comfortable with the concepts of similar audio recommendation systems based on RAG.
- Using music datasets from [this](https://github.com/Yuan-ManX/ai-audio-datasets?tab=readme-ov-file#m) github repo, create a local RAG to recommend sons based on users preferences. Example dataset from that link could be [this](https://zenodo.org/records/5794629) Artificial multitrack audio data. Feel free to find you're own datasets online, or combine the dataset used in this lab with a few you found to make some recommendations.
- Go ahead and build something great in 4 hours.

This notebook uses a local NumPy vector index; no hosted vector database or API credentials are required.

# Install Dependencies

In [ ]:
# Run once if these imports are missing:
# %pip install -q panns-inference datasets

# Load Dataset

In this demo, we will use audio from the *ESC-50 dataset* — a labeled collection of 2000 environmental audio recordings, which are 5-second-long each. The dataset can be loaded from the HuggingFace model hub as follows:

In [6]:
from datasets import load_dataset

# load the dataset from huggingface model hub
data = load_dataset("ashraq/esc50", split="train")
data

Repo card metadata block was not found. Setting CardData to empty.


Generating train split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['filename', 'fold', 'target', 'category', 'esc10', 'src_file', 'take', 'audio'],
    num_rows: 2000
})

The audios in the dataset are sampled at 44100Hz and loaded into NumPy arrays. Let's take a look.

In [20]:
# select the audio feature and display top three
audios = data["audio"]
audios[:3]

[{'path': None,
  'array': array([0., 0., 0., ..., 0., 0., 0.]),
  'sampling_rate': 44100},
 {'path': None,
  'array': array([-0.01184082, -0.10336304, -0.14141846, ...,  0.06985474,
          0.04049683,  0.00274658]),
  'sampling_rate': 44100},
 {'path': None,
  'array': array([-0.00695801, -0.01251221, -0.01126099, ...,  0.215271  ,
         -0.00875854, -0.28903198]),
  'sampling_rate': 44100}]

We only need the Numpy arrays as these contain all of the audio data. We will later input these Numpy arrays directly into our embedding model to generate audio embeddings.

In [23]:
import numpy as np

# select only the audio data from the dataset and store in a numpy array
audios = np.array([a["array"] for a in data["audio"]])

# Load Audio Embedding Model

We will use an audio tagging model trained from *PANNs: Large-Scale Pretrained Audio Neural Networks for Audio Pattern Recognition* paper to generate our audio embeddings. We use the *panns_inference* Python package, which provides an easy interface to load and use the model.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import torch

panns_data = Path.home() / "panns_data"
panns_data.mkdir(parents=True, exist_ok=True)
labels_path = panns_data / "class_labels_indices.csv"
if not labels_path.exists():
    urlretrieve("https://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/class_labels_indices.csv", labels_path)

checkpoint_path = panns_data / "Cnn14_mAP=0.431.pth"
if not checkpoint_path.exists() or checkpoint_path.stat().st_size < 300_000_000:
    urlretrieve("https://zenodo.org/record/3987831/files/Cnn14_mAP%3D0.431.pth?download=1", checkpoint_path)

from panns_inference import AudioTagging

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AudioTagging(checkpoint_path=str(checkpoint_path), device=device)


This notebook uses a local NumPy vector index; no hosted vector database or API credentials are required.

In [ ]:
embedding_dimension = 2048

Create the index:

In [ ]:
embeddings = np.empty((len(audios), embedding_dimension), dtype=np.float32)
embeddings.shape

# Generate Embeddings and Store Locally

Generate audio embeddings in batches to limit memory use, then normalize each vector for cosine-similarity search.

In [ ]:
from tqdm.auto import tqdm

batch_size = 64

for i in tqdm(range(0, len(audios), batch_size)):
    i_end = min(i + batch_size, len(audios))
    batch = audios[i:i_end]
    _, batch_embeddings = model.inference(batch)
    batch_embeddings = np.asarray(batch_embeddings, dtype=np.float32)
    norms = np.linalg.norm(batch_embeddings, axis=1, keepdims=True)
    embeddings[i:i_end] = batch_embeddings / np.maximum(norms, np.finfo(np.float32).eps)

print(f"Indexed {len(embeddings)} audio recordings with {embeddings.shape[1]}-dimensional embeddings.")

The dataset embeddings are now stored locally and ready for cosine-similarity search.

# Querying

This notebook uses a local NumPy vector index; no hosted vector database or API credentials are required.

In [84]:
from IPython.display import Audio, display

# we set an audio number to select from the dataset
audio_num = 400
# get the audio data of the audio number
query_audio = data[audio_num]["audio"]["array"]
# get the category of the audio number
category = data[audio_num]["category"]
# print the category and play the audio
print("Query Audio:", category)
Audio(query_audio, rate=44100)

Query Audio: car_horn


We have got the sound of a car horn. Let's generate an embedding for this sound.

In [ ]:
# reshape query audio
query_audio = query_audio[None, :]
_, xq = model.inference(query_audio)
xq = np.asarray(xq, dtype=np.float32)
xq /= np.maximum(np.linalg.norm(xq, axis=1, keepdims=True), np.finfo(np.float32).eps)
xq.shape

We have now converted the audio into a normalized 2048-dimensional vector. We can compare it against the locally stored embeddings using cosine similarity.

In [ ]:
scores = embeddings @ xq[0]
top_ids = np.argsort(scores)[-3:][::-1]
results = {"matches": [{"id": str(i), "score": float(scores[i])} for i in top_ids]}
results

Notice that the top result is the audio number 400 from our dataset, which is our query audio (the most similar item should always be the query itself). Let's listen to the top three results. 

In [ ]:
# play the top 3 similar audios
for r in results["matches"]:
    # select the audio data from the databse using the id as an index
    a = data[int(r["id"])]["audio"]["array"]
    display(Audio(a, rate=44100))

The results are ranked by cosine similarity. The helper below reuses the already computed embeddings instead of embedding the query again.

In [ ]:
def find_similar_audios(audio_id):
    print("Query Audio:")
    query_audio = data[audio_id]["audio"]["array"]
    display(Audio(query_audio, rate=44100))

    scores = embeddings @ embeddings[audio_id]
    top_ids = np.argsort(scores)[-5:][::-1]
    print("Most similar recordings:")
    for result_id in top_ids:
        audio = data[int(result_id)]["audio"]["array"]
        display(Audio(audio, rate=44100))

In [ ]:
find_similar_audios(1642)

Here we return a set of revving motors (they seem to either be vehicles or lawnmowers).

In [ ]:
find_similar_audios(452)

And now a more relaxing set of birds chirping in nature.

Let's use another audio sample from elsewhere (eg not this dataset) and see how the search performs with this.

In [120]:
#!wget https://storage.googleapis.com/audioset/miaow_16k.wav

We can load the audio into a Numpy array as follows:

In [122]:
import librosa

a, _ = librosa.load("./data/miaow_16k.wav", sr=44100)
Audio(a, rate=44100)

Now we generate an embedding for this external audio and search the local NumPy vector index.

In [ ]:
query_audio = a[None, :]
_, xq = model.inference(query_audio)
xq = np.asarray(xq, dtype=np.float32)
xq /= np.maximum(np.linalg.norm(xq, axis=1, keepdims=True), np.finfo(np.float32).eps)
scores = embeddings @ xq[0]
top_ids = np.argsort(scores)[-3:][::-1]
results = {"matches": [{"id": str(i), "score": float(scores[i])} for i in top_ids]}

for result in results["matches"]:
    audio = data[int(result["id"])]["audio"]["array"]
    display(Audio(audio, rate=44100))

Our audio search application has identified a set of similar cat sounds, which is excellent.

# Clear the Local Index

Clear the in-memory embedding matrix when you are finished querying.

In [ ]:
del embeddings
print("The in-memory audio index has been cleared.")